# `query_entity` — User Guide

Query the financial news pipeline by **entity** (company, person, organisation)
rather than by sector. Entities are extracted dynamically by the LLM — not from
a fixed list — so lookup is always case-insensitive.

**No API key required.** All functions read from `data/sector_summary.tsv`.

```
pipeline/query_entity.py
│
├── Discovery
│   └── list_entities()                                   → list[str]
│
├── Point queries
│   ├── get_entity_snapshot(entity)                       → dict
│   └── get_entity_time_series(entity, lookback_days=30,
│                               include_articles=False)   → dict
│
└── Bulk export
    ├── get_all_entities_ts(lookback_days=None)           → pd.DataFrame
    └── export_entity_ts(output_path=None,
                          lookback_days=90)               → Path
```

| Function | Needs API key | Best for |
|---|---|---|
| `list_entities` | no | discovering what entities are in the data |
| `get_entity_snapshot` | no | "what is the current read on Nvidia?" |
| `get_entity_time_series` | no | "how has sentiment around the Fed changed?" |
| `get_all_entities_ts` | no | cross-entity analysis, filtering, pivoting |
| `export_entity_ts` | no | writing a TSV for R or downstream tools |

## Sections
1. [Setup](#1-setup)
2. [Discover entities](#2-discover-entities--list_entities)
3. [Entity snapshot](#3-entity-snapshot--get_entity_snapshot)
4. [Entity time series](#4-entity-time-series--get_entity_time_series)
5. [Bulk DataFrame](#5-bulk-dataframe--get_all_entities_ts)
6. [Export to TSV](#6-export-to-tsv--export_entity_ts)
7. [End-to-end: cross-entity comparison](#7-end-to-end-cross-entity-comparison)
8. [Error handling reference](#8-error-handling-reference)

---
## 1 — Setup

Adds the project root to `sys.path` so imports work when the notebook is run
from the `notebooks/` subdirectory. All five public functions are imported here
and reused throughout every section.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("..") if Path("../pipeline").exists() else Path(".")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.query_entity import (
    list_entities,
    get_entity_snapshot,
    get_entity_time_series,
    get_all_entities_ts,
    export_entity_ts,
)
from pipeline.constants import SECTOR_SUMMARY_FILE, ENTITY_TS_FILE

print(f"sector_summary.tsv : {SECTOR_SUMMARY_FILE}")
print(f"entity_ts output   : {ENTITY_TS_FILE}")

# Pull sector_summary.tsv from HuggingFace if it doesn't exist locally.
# lacetohf/sector-analysis is public — no token required.
if not SECTOR_SUMMARY_FILE.exists():
    print("\nsector_summary.tsv not found — pulling from HuggingFace ...")
    from dotenv import load_dotenv
    from datasets import load_dataset

    load_dotenv(PROJECT_ROOT / ".env")
    hf_repo  = os.environ.get("HUGGINGFACE_REPO", "lacetohf/feeds")
    hf_user  = hf_repo.split("/")[0]
    repo_id  = f"{hf_user}/sector-analysis"

    ds = load_dataset(repo_id, split="train")
    df_hf = ds.to_pandas()

    SECTOR_SUMMARY_FILE.parent.mkdir(parents=True, exist_ok=True)
    df_hf.to_csv(SECTOR_SUMMARY_FILE, sep="\t", index=False)
    print(f"Saved {len(df_hf):,} rows across {df_hf['date'].nunique()} dates -> {SECTOR_SUMMARY_FILE}")

print(f"\nFile exists : {SECTOR_SUMMARY_FILE.exists()}")

---
## Configuration

Change these values to explore different entities and windows.
All cells below read from these variables — no need to edit them individually.

In [ ]:
ENTITY   = "Nvidia"   # entity to query (case-insensitive)
KEYWORD  = "fed"      # keyword for entity search in section 2
LOOKBACK = 90         # rolling window in days for time-series queries
TOP_N    = 12         # number of top entities to show in the cross-entity heatmap

---
## 2 — Discover entities — `list_entities`

Returns every unique entity name seen across all dates in `sector_summary.tsv`,
sorted alphabetically. Use this to find the correct spelling before calling
`get_entity_snapshot` or `get_entity_time_series`.

**Key behaviour:**
- Returns `[]` gracefully if the TSV is missing — it never raises.
- Canonical casing is the **first spelling** the LLM used (oldest date).
- All other public functions accept any casing — lookup is case-insensitive.

In [ ]:
all_entities = list_entities()

print(f"Total unique entities : {len(all_entities)}")
print(f"\nFirst 20 (alphabetical):")
for name in all_entities[:20]:
    print(f"  {name}")

In [ ]:
# Search for a keyword within the entity list — useful when you're unsure of the exact name.
matches = [e for e in all_entities if KEYWORD.lower() in e.lower()]
print(f"Entities containing '{KEYWORD}': {matches}")

---
## 3 — Entity snapshot — `get_entity_snapshot`

Returns the most recent single-day read on an entity: which sectors mentioned
it, what sentiment each sector recorded, and how old the data is.

**When to use:** "What is the current market read on Nvidia?"

**Return schema:**

| Key | Type | Description |
|---|---|---|
| `entity` | str | canonical casing from the TSV |
| `last_date` | str | YYYY-MM-DD of the most recent mention |
| `data_age_days` | int | days since `last_date` |
| `dominant_sentiment` | str | most frequent sentiment across sectors that day |
| `mean_sentiment_score` | float | mean of −1/0/+1 scores |
| `sectors` | list[dict] | one entry per sector that mentioned the entity |

**Invariant:** `last_date` is the latest calendar date the entity appears in
*any* sector — it may differ from today if the pipeline has not run yet today.

In [ ]:
snap = get_entity_snapshot(ENTITY)

print(f"Entity             : {snap['entity']}")
print(f"Last date          : {snap['last_date']}")
print(f"Data age           : {snap['data_age_days']} days")
print(f"Dominant sentiment : {snap['dominant_sentiment']}")
print(f"Mean score         : {snap['mean_sentiment_score']:.3f}")
print(f"\nSectors ({len(snap['sectors'])} on {snap['last_date']}):")
print(f"  {'sector':<28} {'sentiment':<12} {'score':>5}  {'news_category'}")
print(f"  {'-'*65}")
for s in snap["sectors"]:
    print(f"  {s['sector']:<28} {s['sentiment']:<12} {s['sentiment_score']:>5}  {s['news_category']}")

---
## 4 — Entity time series — `get_entity_time_series`

Returns the sentiment trend for an entity over a rolling window of calendar
days. Each row in `time_series` is one (date × sector) mention.

**When to use:** "How has sentiment around the Fed changed over the last 90 days?"

**Parameters:**

| Parameter | Default | Description |
|---|---|---|
| `entity` | required | entity name, case-insensitive |
| `lookback_days` | 30 | calendar days to look back from today |
| `include_articles` | False | also fetch raw article headlines from feed files |

**Trend direction** is computed as second-half mean − first-half mean over the window:
- `> +0.20` → `"improving"`
- `< −0.20` → `"deteriorating"`
- otherwise → `"stable"`

In [ ]:
ts = get_entity_time_series(ENTITY, lookback_days=LOOKBACK)

print(f"Entity             : {ts['entity']}")
print(f"Window             : {ts['date_range']['from']} -> {ts['date_range']['to']}")
print(f"Observations       : {ts['n_observations']}  (date x sector rows)")
print(f"Mean score         : {ts['mean_sentiment_score']:.3f}")
print(f"Trend              : {ts['trend_direction']}  (delta={ts['trend_delta']:+.3f})")
print(f"Dominant sentiment : {ts['dominant_sentiment']}")
print(f"Sectors seen       : {ts['sectors_seen']}")
print(f"\nCategory breakdown : {ts['category_breakdown']}")
print(f"\nLast 5 time-series rows:")
print(f"  {'date':<12} {'sector':<28} {'sentiment':<12} {'score':>5}  {'news_category'}")
print(f"  {'-'*72}")
for row in ts["time_series"][-5:]:
    print(f"  {row['date']:<12} {row['sector']:<28} {row['sentiment']:<12} {row['sentiment_score']:>5}  {row['news_category']}")

In [ ]:
# Plot the daily sentiment score for this entity over the window.
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

if ts["time_series"]:
    df_ts = pd.DataFrame(ts["time_series"])
    df_ts["date"] = pd.to_datetime(df_ts["date"])

    # Aggregate to one score per date (mean across sectors that day).
    daily = df_ts.groupby("date")["sentiment_score"].mean().sort_index()

    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ["#27ae60" if s > 0 else "#c0392b" if s < 0 else "#95a5a6" for s in daily]
    ax.bar(daily.index, daily.values, color=colors, alpha=0.6, label="Daily mean score")
    ax.plot(daily.index, daily.rolling(7, min_periods=1).mean(),
            color="#2c3e50", linewidth=2, label="7-day rolling mean")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
    plt.xticks(rotation=45, ha="right")
    ax.set_ylabel("Sentiment score  (-1 / 0 / +1)")
    ax.set_title(f"{ts['entity']} — sentiment trend ({ts['lookback_days']}-day window)  |  {ts['trend_direction']}")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No time-series data to plot.")

In [ ]:
# include_articles=True also returns raw headlines for every date in the window.
# Articles come from output/feeds{date}.txt — missing files are skipped with a warning.
ts_with_articles = get_entity_time_series(ENTITY, lookback_days=LOOKBACK, include_articles=True)

articles = ts_with_articles.get("articles") or []
print(f"Articles fetched : {len(articles)}")
if articles:
    print(f"\nSample (first 3):")
    for a in articles[:3]:
        print(f"  [{a['date']}] {a['title'][:80]}")

---
## 5 — Bulk DataFrame — `get_all_entities_ts`

Returns a long-format `pd.DataFrame` with **one row per (date × entity × sector)**
mention across all entities in the TSV. This is the raw material for any
custom cross-entity analysis.

**Columns:** `date`, `entity`, `sector`, `sentiment`, `sentiment_score`, `news_category`

**When to use:**
- Comparing multiple entities at once
- Building a pivot table (entities × dates)
- Filtering by sector or news category across all entities
- Feeding into downstream analytics or visualisations

Pass `lookback_days=N` to restrict to a rolling window; omit it (or pass `None`)
for the full history.

In [ ]:
df_all = get_all_entities_ts(lookback_days=LOOKBACK)

print(f"Shape      : {df_all.shape}  (rows x cols)")
print(f"Columns    : {list(df_all.columns)}")
print(f"Entities   : {df_all['entity'].nunique()} unique")
print(f"Date range : {df_all['date'].min().date()} -> {df_all['date'].max().date()}")
print(f"\nSample rows:")
df_all.head(6)

In [ ]:
# Top 20 most-mentioned entities in the window — ranked by total row count.
top_entities = (
    df_all.groupby("entity")
    .agg(
        mentions=("sentiment_score", "count"),
        mean_score=("sentiment_score", "mean"),
        sectors=("sector", "nunique"),
    )
    .sort_values("mentions", ascending=False)
    .head(20)
    .round(3)
)

print(f"Top 20 entities by mention count (last 90 days):\n")
print(f"{'entity':<30} {'mentions':>8}  {'mean_score':>10}  {'sectors':>7}")
print("-" * 60)
for entity, row in top_entities.iterrows():
    bar = "#" * int((row["mean_score"] + 1) * 5)  # scale -1..+1 to 0..10
    print(f"{entity:<30} {row['mentions']:>8}  {row['mean_score']:>10.3f}  {row['sectors']:>7}  {bar}")

---
## 6 — Export to TSV — `export_entity_ts`

Writes the long-format entity time series to a TSV file and returns the path.

- Default output: `data/entity_sentiment_ts.tsv` (controlled by `ENTITY_TS_FILE`)
- Default window: 90 days (`EXPORT_LOOKBACK_DAYS`)
- The `date` column is written as plain `YYYY-MM-DD` (no timestamp) — R-friendly

Pass `output_path` to write to a custom location. Pass `lookback_days=None` for
the full history. This is the same file pushed to `lacetohf/entity-sentiment`
by the daily CI workflow.

In [ ]:
import tempfile

# Write to a temp file so we don't overwrite the live export during exploration.
with tempfile.NamedTemporaryFile(suffix=".tsv", delete=False) as f:
    tmp_path = f.name

out_path = export_entity_ts(output_path=tmp_path, lookback_days=90)

# Verify the file.
import pandas as pd
df_exported = pd.read_csv(out_path, sep="\t")

print(f"Written to  : {out_path}")
print(f"Shape       : {df_exported.shape}")
print(f"Columns     : {list(df_exported.columns)}")
print(f"Date dtype  : {df_exported['date'].dtype}  (should be object/str for R)")
print(f"\nFirst 3 rows:")
print(df_exported.head(3).to_string(index=False))

---
## 7 — End-to-end: cross-entity comparison

Combines `get_all_entities_ts` with a pivot and a heatmap to answer:
**"Which of the top entities have the most positive/negative sentiment trend?"**

1. Load the bulk DataFrame for the last 90 days.
2. Keep only the top-N most-mentioned entities.
3. Pivot to a (date × entity) mean-score matrix.
4. Plot as a diverging heatmap — green = positive, red = negative, grey = missing.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df_bulk = get_all_entities_ts(lookback_days=LOOKBACK)

if df_bulk.empty:
    print("No entity data available — run the pipeline first.")
else:
    # 1. Pick top-N by mention frequency.
    top_names = (
        df_bulk.groupby("entity")["sentiment_score"]
        .count()
        .nlargest(TOP_N)
        .index.tolist()
    )

    # 2. Filter and aggregate to daily mean per entity.
    df_top = df_bulk[df_bulk["entity"].isin(top_names)].copy()
    daily_pivot = (
        df_top.groupby(["date", "entity"])["sentiment_score"]
        .mean()
        .unstack("entity")
        .sort_index()
        .reindex(columns=top_names)
    )

    # 3. Print summary table — overall mean per entity.
    print(f"Cross-entity sentiment summary (last {LOOKBACK} days):\n")
    print(f"{'entity':<30} {'mean_score':>10}  {'direction'}")
    print("-" * 55)
    for col in daily_pivot.columns:
        scores = daily_pivot[col].dropna()
        if len(scores) < 2:
            direction = "n/a"
        else:
            mid = len(scores) // 2
            delta = scores.iloc[mid:].mean() - scores.iloc[:mid].mean()
            direction = "improving" if delta > 0.20 else "deteriorating" if delta < -0.20 else "stable"
        print(f"{col:<30} {scores.mean():>10.3f}  {direction}")

    # 4. Heatmap — dates on y-axis, entities on x-axis.
    fig, ax = plt.subplots(figsize=(max(10, TOP_N * 1.1), 6))
    im = ax.imshow(
        daily_pivot.values,
        aspect="auto",
        cmap="RdYlGn",
        vmin=-1, vmax=1,
        interpolation="nearest",
    )
    ax.set_xticks(range(len(daily_pivot.columns)))
    ax.set_xticklabels(daily_pivot.columns, rotation=45, ha="right", fontsize=9)

    n_dates = len(daily_pivot)
    step = max(1, n_dates // 10)
    y_ticks = list(range(0, n_dates, step))
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(
        [str(daily_pivot.index[i].date()) for i in y_ticks], fontsize=8
    )

    plt.colorbar(im, ax=ax, label="Mean sentiment score  (-1 negative / +1 positive)")
    ax.set_title(f"Entity sentiment heatmap — top {TOP_N} by mentions, last {LOOKBACK} days")
    plt.tight_layout()
    plt.show()

---
## 8 — Error handling reference

### Guard conditions per function

| Function | Guard condition | Error raised |
|---|---|---|
| `get_entity_snapshot` | entity not in TSV (case-insensitive) | `LookupError` with similar-name hints |
| `get_entity_snapshot` | TSV missing or empty | `LookupError` |
| `get_entity_time_series` | entity not in TSV | `LookupError` with similar-name hints |
| `get_entity_time_series` | entity has no data within `lookback_days` window | `LookupError` |
| `export_entity_ts` | TSV missing or no data in window | `RuntimeError` |

`list_entities` and `get_all_entities_ts` **never raise** — they return `[]`
and an empty DataFrame respectively, each with a `warnings.warn`.

The `LookupError` message always includes up to 10 similar names (prefix or
substring matches) to help you find the correct spelling.

In [ ]:
# Entity not found — includes similar-name hints in the message.
try:
    get_entity_snapshot("nvdia")   # deliberate typo
except LookupError as e:
    print(f"LookupError: {e}")

In [ ]:
# Entity exists but has no data within a very short window.
try:
    get_entity_time_series(ENTITY, lookback_days=1)
except LookupError as e:
    print(f"LookupError: {e}")

In [ ]:
# export_entity_ts raises RuntimeError when the result would be empty.
import tempfile, os

try:
    # lookback_days=1 will almost certainly produce an empty result.
    export_entity_ts(
        output_path=os.path.join(tempfile.gettempdir(), "test_empty.tsv"),
        lookback_days=1,
    )
except RuntimeError as e:
    print(f"RuntimeError: {e}")